# Topic Modeling

## Load the dataset

In [18]:
import pandas as pd

exploratory_df = pd.read_csv("dataset/processed/exploratory-packages-feature-engineering.csv")
confirmatory_df = pd.read_csv("dataset/processed/confirmatory-packages-feature-engineering.csv")
holdout_df = pd.read_csv("dataset/processed/holdout-packages-feature-engineering.csv")

exploratory_df.head()

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,first_place,...,neg,neu,pos,compound,created_at_dayofweek,created_at_hourofday,test_group_size,read_flesch,read_coleman,specificity_tfidf
0,0,2014-11-20 06:43:16.005,201446,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,546d6fa19ad54eec8d00002d,3052,150,0.049148,True,...,0.218,0.479,0.303,0.1531,3,6,4,89.606731,8.2,0.350177
1,1,2014-11-20 06:43:44.646,201446,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,546d6fa19ad54eec8d00002d,3033,122,0.040224,False,...,0.218,0.479,0.303,0.1531,3,6,4,89.606731,8.2,0.350177
2,2,2014-11-20 06:44:59.804,201446,546d88fb84ad38b2ce000024,They're Being Called 'Walmart's Worst Nightmar...,546d6fa19ad54eec8d00002d,3092,110,0.035576,False,...,0.218,0.479,0.303,0.1531,3,6,4,89.606731,8.2,0.350177
3,3,2014-11-20 06:54:36.335,201446,546d902c26714c6c44000039,This Is What Sexism Against Men Sounds Like,546bc55335992b86c8000043,3526,90,0.025525,False,...,0.000,0.737,0.263,0.3612,3,6,8,82.390000,6.6,0.486990
4,4,2014-11-20 06:54:57.878,201446,546d902c26714c6c44000039,This Is What Sexism Against Men Sounds Like,546d900426714cd2dd00002e,3506,120,0.034227,True,...,0.000,0.737,0.263,0.3612,3,6,8,82.390000,6.6,0.486990


## BERTopic

I decided to use the `all-MiniLM-L6-v2` model for the embeddings as it's a simple model for this task with a good performance track and recommended by the library documentation.

In [19]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

exploratory_headlines = exploratory_df["headline"].fillna("").tolist()
confirmatory_headlines = confirmatory_df["headline"].fillna("").tolist()
holdout_headlines = holdout_df["headline"].fillna("").tolist()

# Create the SentenceTransformer embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Fit the BERTTopic model
# Set min_topic_size so only topics with at least 80 headlines are considered
# For IF-IDF I use unigrams and bigrams
topic_model = BERTopic(
    embedding_model=embedding_model,
    n_gram_range=(1, 2),
    min_topic_size=80,
    top_n_words=20,
    verbose=True,
)

exploratory_topics, exploratory_probs = topic_model.fit_transform(exploratory_headlines)

exploratory_df["topic_bertopic"] = exploratory_topics

# Use the fitted model to assign topics to confirmatory and holdout sets
confirmatory_headlines = confirmatory_df["headline"].fillna("").tolist()
holdout_headlines = holdout_df["headline"].fillna("").tolist()
confirmatory_topics, confirmatory_probs = topic_model.transform(confirmatory_headlines)
holdout_topics, holdout_probs = topic_model.transform(holdout_headlines)
confirmatory_df["topic_bertopic"] = confirmatory_topics
holdout_df["topic_bertopic"] = holdout_topics

exploratory_df[["headline", "topic_bertopic"]].head()


2025-12-06 13:06:37,354 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/709 [00:00<?, ?it/s]

2025-12-06 13:07:04,004 - BERTopic - Embedding - Completed ✓
2025-12-06 13:07:04,004 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-06 13:07:09,963 - BERTopic - Dimensionality - Completed ✓
2025-12-06 13:07:09,968 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current p

Batches:   0%|          | 0/3299 [00:00<?, ?it/s]

2025-12-06 13:09:09,194 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-12-06 13:09:26,849 - BERTopic - Dimensionality - Completed ✓
2025-12-06 13:09:26,851 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-12-06 13:09:32,138 - BERTopic - Cluster - Completed ✓


Batches:   0%|          | 0/707 [00:00<?, ?it/s]

2025-12-06 13:09:59,603 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-12-06 13:10:02,470 - BERTopic - Dimensionality - Completed ✓
2025-12-06 13:10:02,470 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-12-06 13:10:03,538 - BERTopic - Cluster - Completed ✓


,headline,topic_bertopic
0,They're Being Called 'Walmart's Worst Nightmar...,-1
1,They're Being Called 'Walmart's Worst Nightmar...,-1
2,They're Being Called 'Walmart's Worst Nightmar...,-1
3,This Is What Sexism Against Men Sounds Like,-1
4,This Is What Sexism Against Men Sounds Like,-1


In [20]:
# Print the number of headlines per topic
print(f"Exploratory set: Number of headlines per topic ({len(exploratory_df)} headlines):")
print(exploratory_df["topic_bertopic"].value_counts().head(20))

print(f"Confirmatory set: Number of headlines per topic ({len(confirmatory_df)} headlines):")
print(confirmatory_df["topic_bertopic"].value_counts().head(20))

print(f"Holdout set: Number of headlines per topic ({len(holdout_df)} headlines):")
print(holdout_df["topic_bertopic"].value_counts().head(20))


Exploratory set: Number of headlines per topic (22666 headlines):
topic_bertopic
-1     11610
 0      1869
 1       902
 2       658
 3       629
 4       575
 5       555
 6       474
 7       365
 8       324
 9       314
 10      274
 11      248
 12      248
 13      231
 14      219
 15      198
 16      181
 17      177
 18      170
Name: count, dtype: int64
Confirmatory set: Number of headlines per topic (105551 headlines):
topic_bertopic
-1     50754
 0      9839
 1      5060
 3      3492
 4      3246
 2      3095
 5      2581
 6      2280
 7      1650
 9      1627
 8      1583
 11     1378
 10     1157
 12     1064
 15     1063
 17     1035
 13     1006
 16      827
 18      814
 21      798
Name: count, dtype: int64
Holdout set: Number of headlines per topic (22600 headlines):
topic_bertopic
-1     10747
 0      2117
 1      1108
 4       792
 3       763
 5       651
 2       594
 6       500
 8       361
 9       318
 7       298
 12      290
 11      248
 10      232
 13  

Around half the headlines don't have a topic assigned. This is ok, we only care about the topics we can really identify rather than noise.

The good thing is that the topics discovered using the exploratory set were fitted to the confirmatory and holdout sets in similar proportions.

### Topics Analysis

In [21]:
# Inspect the discovered topics

# Overview of the 20 most common topics
topic_info = topic_model.get_topic_info()
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,11610,-1_the_to_you_it,"[the, to, you, it, of, this, and, in, is, that...","[Here's What It Feels Like To Be Gay, If You A..."
1,0,1869,0_she_her_to_the,"[she, her, to, the, he, was, his, and, but, it...","[Women Like Her, Who Did What She Did, Aren't ..."
2,1,902,1_kids_to_these_teachers,"[kids, to, these, teachers, these kids, of, th...",[The 4 Words You Should Be Saying To Kids Inst...
3,2,658,2_women_feminism_men_feminist,"[women, feminism, men, feminist, they, the, to...","[If You Still Don't Think We Need Feminism, Yo..."
4,3,629,3_gay_straight_marriage_gay marriage,"[gay, straight, marriage, gay marriage, the, t...",[A Pastor Asks A Politician Why He Supports Ga...
5,4,575,4_video_this video_seconds_this,"[video, this video, seconds, this, the, you, d...",[Over 2/3 Of The World Can't Watch This Video....
6,5,555,5_white_race_white people_people,"[white, race, white people, people, comedian, ...",[A New And Creative Way To Help White People U...
7,6,474,6_food_restaurant_you_the,"[food, restaurant, you, the, eat, your, it, fa...",['What's This About Fast Food Workers Complain...
8,7,365,7_news_fox news_fox_ferguson,"[news, fox news, fox, ferguson, the, police, c...",[Fox News finally went off on Walmart for thei...
9,8,324,8_jobs_money_poor_people,"[jobs, money, poor, people, welfare, the, of, ...",[The Top 6 Reasons Why Money Spent On Keeping ...


Looking at the most frequent words in the most common topics we easily interpret what most of them are about:

- –1: Miscellaneous
- 0: She, He, Pronouns
- 1: Kids
- 2: Feminism
- 3: Gay Marriage
- 4: Viral Videos & Content
- 5: Race & White People
- 6: Food & Restaurants
- 7: Fox News
- 8: Jobs & Money
- 9: Music
- 10: Water & City
- 11: Science
- 12: Rape & Sexual Violence
- 13: Abortion & Reproductive Rights
- 14: Football & Sports
- 15: Climate Change
- 16: Space
- 17: Fashion
- 18: Minimum Wage

Now I'll save the dataset

In [22]:
exploratory_df.to_csv("dataset/processed/exploratory-packages-topic-modeling.csv", index=False)
confirmatory_df.to_csv("dataset/processed/confirmatory-packages-topic-modeling.csv", index=False)
holdout_df.to_csv("dataset/processed/holdout-packages-topic-modeling.csv", index=False)

For future reference, I'll save all the topics with the most frequent words.

In [23]:
# Save the topics
topics = topic_model.get_topics()

rows = []
for topic_id, word_weights in topics.items():
    # Take the top 50 words for this topic
    top_words = [word for word, _ in word_weights[:50]]
    rows.append({
        "topic_id": topic_id,
        "top_50_words": ", ".join(top_words),
    })

topics_df = pd.DataFrame(rows).sort_values("topic_id")

output_path = "dataset/processed/topics.csv"
topics_df.to_csv(output_path, index=False)
topics_df.head()

,topic_id,top_50_words
0,-1,"the, to, you, it, of, this, and, in, is, that,..."
1,0,"she, her, to, the, he, was, his, and, but, it,..."
2,1,"kids, to, these, teachers, these kids, of, the..."
3,2,"women, feminism, men, feminist, they, the, to,..."
4,3,"gay, straight, marriage, gay marriage, the, to..."
